In [1]:
import pandas as pd
import numpy as np
import joblib
from scipy.stats import norm


In [2]:
# Load cleaned options dataset
df = pd.read_csv("../data/processed/multi_ticker_options_clean.csv")

# Add binary label again
df["label"] = (df["lastPrice"] > df["bid"]).astype(int)

# Recreate 'moneyness' if it's missing
if "moneyness" not in df.columns:
    df["moneyness"] = df["spot_price"] / df["strike"]

print("✅ Dataset loaded. Shape:", df.shape)


✅ Dataset loaded. Shape: (1869, 21)


In [3]:
# Load your trained LightGBM model
model = joblib.load("models/final_lightgbm_with_quant_features.pkl")  # Or whatever name you saved it as


In [4]:
def black_scholes_price(S, K, T, r, sigma, option_type="call"):
    if sigma <= 0 or T <= 0:
        return max(S - K, 0) if option_type == "call" else max(K - S, 0)

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


In [5]:
r = 0.05  # constant risk-free rate assumption

df["bs_price"] = df.apply(lambda row: black_scholes_price(
    row["spot_price"],
    row["strike"],
    row["days_to_expiry"] / 365,
    r,
    row["impliedVolatility"]
), axis=1)

print("✅ Black-Scholes pricing complete.")


✅ Black-Scholes pricing complete.


In [6]:
def monte_carlo_price(S, K, T, r, sigma, sims=10000, option_type="call"):
    if sigma <= 0 or T <= 0:
        return max(S - K, 0) if option_type == "call" else max(K - S, 0)

    np.random.seed(0)
    Z = np.random.standard_normal(sims)
    ST = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    if option_type == "call":
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)

    return np.exp(-r * T) * np.mean(payoffs)


In [7]:
df["mc_price"] = df.apply(lambda row: monte_carlo_price(
    row["spot_price"],
    row["strike"],
    row["days_to_expiry"] / 365,
    r,
    row["impliedVolatility"]
), axis=1)

print("✅ Monte Carlo prices added.")


✅ Monte Carlo prices added.


In [8]:
df["bs_diff"] = df["lastPrice"] - df["bs_price"]
df["mc_diff"] = df["lastPrice"] - df["mc_price"]
df["bs_vs_mc"] = df["bs_price"] - df["mc_price"]

print("✅ Quant finance features created.")


✅ Quant finance features created.


In [9]:
df[["lastPrice", "bs_price", "mc_price", "bs_diff", "mc_diff", "bs_vs_mc"]].head()


,lastPrice,bs_price,mc_price,bs_diff,mc_diff,bs_vs_mc
0,252.03,269.909973,269.909973,-17.879973,-17.879973,0.0
1,263.21,264.909973,264.909973,-1.699973,-1.699973,0.0
2,235.73,239.909973,239.909973,-4.179973,-4.179973,0.0
3,200.19,234.909973,234.909973,-34.719973,-34.719973,0.0
4,220.98,229.909973,229.909973,-8.929973,-8.929973,0.0


In [13]:
features = df.select_dtypes(include=[np.number]).drop(columns=["label"], errors="ignore")


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
features_scaled = pd.DataFrame(features_scaled, columns=features.columns)


In [15]:
df["predicted_proba"] = model.predict_proba(features_scaled)[:, 1]
print("✅ Model probabilities added.")


✅ Model probabilities added.


In [16]:
df["signal"] = (df["predicted_proba"] > 0.6).astype(int)


In [18]:
df["pnl"] = np.zeros (len(df), dtype=float)  # default: no trade
df.loc[df["signal"] == 1, "pnl"] = df["lastPrice"] - df["bid"]


In [19]:
num_trades = df["signal"].sum()
total_profit = df["pnl"].sum()
avg_pnl = df.loc[df["signal"] == 1, "pnl"].mean()
win_rate = (df.loc[df["signal"] == 1, "pnl"] > 0).mean()

print(f"📊 Number of trades: {num_trades}")
print(f"💰 Total PnL: {total_profit:.2f}")
print(f"📈 Average PnL per trade: {avg_pnl:.2f}")
print(f"✅ Win rate: {win_rate:.2%}")


📊 Number of trades: 1145
💰 Total PnL: 1356.41
📈 Average PnL per trade: 1.18
✅ Win rate: 98.86%


In [20]:
def simulate_expiry_payoff(S, K, T, r, sigma, cost, sims=10000, option_type="call"):
    """
    S: current spot price
    K: strike price
    T: time to expiry (in years)
    r: risk-free rate
    sigma: implied volatility
    cost: what you pay (bid)
    """
    if sigma <= 0 or T <= 0:
        return max(S - K, 0) - cost if option_type == "call" else max(K - S, 0) - cost

    np.random.seed(0)
    Z = np.random.standard_normal(sims)
    ST = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    if option_type == "call":
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)

    expected_payoff = np.exp(-r * T) * np.mean(payoffs)
    return expected_payoff - cost  # subtract your entry price (bid)


In [21]:
r = 0.05  # constant risk-free rate

df["simulated_pnl_expiry"] = df.apply(lambda row: simulate_expiry_payoff(
    S=row["spot_price"],
    K=row["strike"],
    T=row["days_to_expiry"] / 365,
    r=r,
    sigma=row["impliedVolatility"],
    cost=row["bid"],  # what you paid to buy
    option_type="call"  # assuming all are calls for now
), axis=1)

print("✅ Simulated expiry PnL calculated.")


✅ Simulated expiry PnL calculated.


In [22]:
df["simulated_pnl_expiry"].describe()


count    1869.000000
mean        0.545884
std         4.910086
min        -5.080723
25%        -0.258397
50%         0.000000
75%         1.395474
max       193.090027
Name: simulated_pnl_expiry, dtype: float64